In [1]:
import os
import polars as pl
from tqdm.notebook import tqdm
from optuna.samplers import TPESampler
import optuna

In [2]:
# pl.Config.set_tbl_rows(50)
# pl.Config.set_fmt_float("full")

In [3]:
from __future__ import annotations
from pathlib import Path
from typing import Iterable, Dict, List, Optional, Union
import json
import polars as pl

class Tokenizer:
    def __init__(
        self,
        codes_parquet_fp: str | Path,
        special_tokens: Optional[Iterable[str]] = None,
        force_special_ids: bool = True,  # pin [PAD]=0 etc.
    ):
        if special_tokens is None:
            special_tokens = ["[PAD]", "[MASK]", "[CLS]", "[UNK]"]


        df_codes = pl.read_parquet(str(codes_parquet_fp), columns=["code"])
        base_codes = df_codes.get_column("code").to_list()
        seen = set()
        unique_codes = []
        for c in base_codes:
            if c not in seen:
                unique_codes.append(c)
                seen.add(c)

        vocab_list: List[str] = []
        special_tokens = list(special_tokens)

        if force_special_ids:
            for tok in ["[PAD]", "[MASK]", "[CLS]", "[UNK]"]:
                if tok in special_tokens and tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)
                elif tok in special_tokens and tok in seen:

                    vocab_list.append(tok)
            for tok in special_tokens:
                if tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)
        else:
            for tok in special_tokens:
                if tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)


        vocab_list.extend(unique_codes)


        self.id2code: List[str] = vocab_list
        self.code2id: Dict[str, int] = {tok: idx for idx, tok in enumerate(vocab_list)}
        self.vocab_size: int = len(self.id2code)

        self.pad_token  = "[PAD]" if "[PAD]" in self.code2id else None
        self.mask_token = "[MASK]" if "[MASK]" in self.code2id else None
        self.cls_token  = "[CLS]" if "[CLS]" in self.code2id else None
        self.unk_token  = "[UNK]" if "[UNK]" in self.code2id else None

        self.pad_id  = self.code2id[self.pad_token]  if self.pad_token  else 0
        self.mask_id = self.code2id[self.mask_token] if self.mask_token else None
        self.cls_id  = self.code2id[self.cls_token]  if self.cls_token  else None
        self.unk_id  = self.code2id[self.unk_token]  if self.unk_token  else None


        type_set = set()
        for tok in self.id2code:
            prefix = tok.split("//", 1)[0]
            type_set.add(prefix)

        types_sorted = sorted(t for t in type_set if t not in ("[PAD]",))
        self.type2id: Dict[str, int] = {"[PAD]": 0}
        next_id = 1
        for sp in ["[MASK]", "[CLS]", "[UNK]"]:
            if sp in type_set:
                self.type2id[sp] = next_id; next_id += 1
        for t in types_sorted:
            if t not in self.type2id:
                self.type2id[t] = next_id
                next_id += 1

        self._code2id_df = pl.DataFrame({"code": self.id2code,
                                         "input_id": list(range(self.vocab_size))}) \
                               .with_columns(pl.col("code").cast(pl.Categorical))
        self._type2id_df = pl.DataFrame({"code_type": list(self.type2id.keys()),
                                         "type_id":   list(self.type2id.values())}) \
                               .with_columns(pl.col("code_type").cast(pl.Categorical))


    def encode(self, codes: Iterable[str]) -> List[int]:
        get = self.code2id.get
        if self.unk_id is not None:
            fallback = self.unk_id
        else:
            fallback = self.pad_id if self.pad_id is not None else 0
        return [get(c, fallback) for c in codes]

    def decode(self, ids: Iterable[int]) -> List[str]:
        out = []
        for i in ids:
            if 0 <= i < self.vocab_size:
                out.append(self.id2code[i])
            else:
                out.append(self.unk_token or "[UNK]")
        return out

    def save(self, path: str | Path) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        obj = {
            "id2code": self.id2code,
            "special_tokens": [t for t in ["[PAD]", "[MASK]", "[CLS]", "[UNK]"] if t in self.code2id],
            "type2id": self.type2id,
            "pad_id": self.pad_id,
            "mask_id": self.mask_id,
            "cls_id": self.cls_id,
            "unk_id": self.unk_id,
        }
        with open(path, "w") as f:
            json.dump(obj, f, indent=2)

    @classmethod
    def load(cls, path: str | Path) -> "Tokenizer":
        path = Path(path)
        with open(path) as f:
            obj = json.load(f)

        tok = cls.__new__(cls) 

        tok.id2code = obj["id2code"]
        tok.code2id = {tok_: i for i, tok_ in enumerate(tok.id2code)}
        tok.vocab_size = len(tok.id2code)

        tok.special_tokens = obj.get("special_tokens", [])
        tok.type2id = obj.get("type2id", {})

        tok.pad_token  = "[PAD]" if "[PAD]" in tok.code2id else None
        tok.mask_token = "[MASK]" if "[MASK]" in tok.code2id else None
        tok.cls_token  = "[CLS]" if "[CLS]" in tok.code2id else None
        tok.unk_token  = "[UNK]" if "[UNK]" in tok.code2id else None

        tok.pad_id  = obj.get("pad_id", tok.code2id.get("[PAD]", 0))
        tok.mask_id = obj.get("mask_id", tok.code2id.get("[MASK]")) if "[MASK]" in tok.code2id else None
        tok.cls_id  = obj.get("cls_id", tok.code2id.get("[CLS]"))   if "[CLS]" in tok.code2id else None
        tok.unk_id  = obj.get("unk_id", tok.code2id.get("[UNK]"))   if "[UNK]" in tok.code2id else None

        # Rebuild Polars lookup frames
        tok._code2id_df = pl.DataFrame({"code": tok.id2code,
                                        "input_id": list(range(tok.vocab_size))}) \
                              .with_columns(pl.col("code").cast(pl.Categorical))
        tok._type2id_df = pl.DataFrame({"code_type": list(tok.type2id.keys()),
                                        "type_id":   list(tok.type2id.values())}) \
                              .with_columns(pl.col("code_type").cast(pl.Categorical))
        return tok

    @property
    def code2id_df(self) -> pl.DataFrame:
        return self._code2id_df

    @property
    def type2id_df(self) -> pl.DataFrame:
        return self._type2id_df

In [4]:
from __future__ import annotations
from math import ceil
from typing import Iterable, Dict, List, Optional, Union, Any, Literal
import polars as pl
import torch

class SequencesGenerator:


    def __init__(
        self,
        tokenizer_path: str,
        chunk_length: int = 1024,
        overlap: int = 128,
        return_numeric: bool = False,
        return_text: bool = False,
        return_time: bool = False,
        return_ids: bool = False,
    ):

        self.tokenizer = Tokenizer.load(tokenizer_path)
        self.chunk_length = chunk_length
        self.overlap = overlap
        self.return_numeric = return_numeric
        self.return_text = return_text
        self.return_time = return_time
        self.return_ids = return_ids

    def encode_sequence(
        self,
        timeline: pl.DataFrame,
        max_length: Optional[int] = None,
        pad_to_max: bool = False,
        truncation: Literal["head", "tail"] = "tail",
        add_cls: bool = False,
    ) -> Dict[str, Union[List[int], List[float], List[str]]]:
        """
        Vectorized build of:
          input_ids, attention_mask, visit_ids, stage_ids, type_ids
          + optional numeric/text streams (+ masks)
        """
        df = timeline
        
        if "seq_id" in df.columns:

            uniq = df.select(pl.col("seq_id")).unique(maintain_order=True)
            uniq = uniq.with_row_count(name="visit_ids_raw")  # 0..K-1
            df = df.join(uniq, on="seq_id", how="left").with_columns(
                (pl.col("visit_ids_raw") ).alias("visit_id").fill_null(0)
            ).drop("visit_ids_raw")
        else:
            df = df.with_columns(pl.lit(0).alias("visit_id"))

        stage_cols = ["out_id", "er_id", "hadm_id", "icustay_id"]
        present_stages = [c for c in stage_cols if c in df.columns]
        if present_stages:

            expr = pl.lit(0)
            for i, col in enumerate(present_stages, start=1):
                expr = pl.when(expr.eq(0) & pl.col(col).is_not_null()).then(i).otherwise(expr)
            df = df.with_columns(expr.alias("stage_id"))
        else:
            df = df.with_columns(pl.lit(0).alias("stage_id"))

        df = df.join(
            self.tokenizer.type2id_df,
            on=pl.col("code_type").cast(pl.Categorical),
            how="left",
        ).with_columns(pl.col("type_id").fill_null(0))

        df = df.join(
            self.tokenizer.code2id_df,
            on=pl.col("code").cast(pl.Categorical),
            how="left",
        )


        unk_id = self.tokenizer.unk_id if self.tokenizer.unk_id is not None else self.tokenizer.pad_id or 0
        df = df.with_columns(pl.col("input_id").fill_null(unk_id))

        df = df.with_columns(
            pl.when(pl.col("code").str.starts_with("TIME-GAP//"))
              .then(0)
              .otherwise(pl.col("visit_id"))
              .alias("visit_id"),
            pl.when(pl.col("code").str.starts_with("TIME-GAP//"))
              .then(0)
              .otherwise(pl.col("stage_id"))
              .alias("stage_id"),
        )


        if add_cls and (self.tokenizer.cls_token is not None):
            cls_row = {
                "code": self.tokenizer.cls_token,
                "code_type": "[CLS]",
                "visit_id": 0,
                "stage_id": 0,
                "type_id": self.tokenizer.type2id.get("[CLS]", 0),
                "input_id": self.tokenizer.cls_id,
            }

            if self.return_numeric:
                cls_row["numeric_value"] = None
            if self.return_text:
                cls_row["text_value"] = None

            df = pl.concat([pl.DataFrame([cls_row]), df], how="vertical_relaxed")


        input_ids = df.get_column("input_id").cast(pl.Int64).to_list()
        type_ids = df.get_column("type_id").cast(pl.Int64).to_list()
        visit_ids = df.get_column("visit_id").cast(pl.Int64).to_list()
        stage_ids = df.get_column("stage_id").cast(pl.Int64).to_list()
        attention_mask = [1] * len(input_ids)


        value_payload = self._build_value_streams(
            df=df,
            max_length=max_length,
            pad_to_max=pad_to_max,
            truncation=truncation,
        )

        # --- truncate/pad core streams in one go ---
        input_ids      = self._truncate(input_ids,      max_length, truncation)
        type_ids       = self._truncate(type_ids,       max_length, truncation)
        visit_ids      = self._truncate(visit_ids,      max_length, truncation)
        stage_ids      = self._truncate(stage_ids,      max_length, truncation)
        attention_mask = [1] * len(input_ids)

        if pad_to_max and max_length is not None and len(input_ids) < max_length:
            pad_len = max_length - len(input_ids)
            pad_id = self.tokenizer.pad_id if self.tokenizer.pad_id is not None else 0
            input_ids      = input_ids + [pad_id] * pad_len
            type_ids       = type_ids + [0] * pad_len
            visit_ids      = visit_ids + [0] * pad_len
            stage_ids      = stage_ids + [0] * pad_len
            attention_mask = attention_mask + [0] * pad_len

        out = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "visit_ids": visit_ids,
            "stage_ids": stage_ids,
            "type_ids": type_ids,
        }
        out.update(value_payload)
        return out

    def get_overlapped_chunks(
        self,
        timeline: Dict[str, Iterable],
        chunk_length: Optional[int] = None,
        overlap: Optional[int] = None,
        add_cls_per_chunk: bool = True,
    ) -> List[Dict[str, List[Any]]]:
        """
        Sliding-window chunking with optional [CLS] per chunk and padding.
        """
        if chunk_length is None or overlap is None:
            chunk_length = self.chunk_length
            overlap = self.overlap

        fields = ["input_ids", "attention_mask", "visit_ids", "stage_ids", "type_ids"]
        for extra in ("numeric_values", "numeric_mask", "text_values", "text_mask", 'time_diff'):
            if extra in timeline and extra not in fields:
                fields.append(extra)

        n = len(timeline["input_ids"])
        payload = chunk_length - (1 if add_cls_per_chunk else 0)
        step = max(1, payload - overlap)

        num_chunks = 1 if n <= payload else ceil((n - payload) / step) + 1
        starts = [i * step for i in range(num_chunks)]

        cls_defaults = {
            "input_ids": self.tokenizer.cls_id if self.tokenizer.cls_id is not None else (self.tokenizer.pad_id or 0),
            "attention_mask": 1,
            "visit_ids": 0,
            "stage_ids": 0,
            "type_ids": self.tokenizer.type2id.get("[CLS]", 0),
            "numeric_values": 0.0,
            "numeric_mask": 0,
            "text_values": "",
            "text_mask": 0,
            "time_diff":0.0}

        chunks = []
        for start in starts:
            end = min(n, start + payload)
            sliced = {k: list(timeline[k][start:end]) for k in fields if k in timeline}

            if "attention_mask" in sliced:
                sliced["attention_mask"] = [1] * len(sliced["input_ids"])

            if add_cls_per_chunk:
                for k in list(sliced.keys()):
                    sliced[k] = [cls_defaults[k]] + sliced[k]

            cur_len = len(sliced["input_ids"])
            if cur_len < chunk_length:
                pad_len = chunk_length - cur_len
                for k in list(sliced.keys()):
                    sliced[k] = self._pad_list(sliced[k], pad_len, 0)

            chunks.append(sliced)
        return chunks


    def _build_value_streams(
        self,
        df: pl.DataFrame,
        max_length: Optional[int],
        pad_to_max: bool,
        truncation: Literal["head", "tail"],
    ) -> Dict[str, List[Any]]:
        out: Dict[str, List[Any]] = {}
        # Numeric stream
        if self.return_numeric:
            if "numeric_value" in df.columns:
                vals = df.get_column("numeric_value").to_list()
            else:
                vals = [None] * df.height
            num_mask = [1 if (v is not None) else 0 for v in vals]
            vals = [0.0 if v is None else float(v) for v in vals]

            vals = self._truncate(vals, max_length, truncation)
            num_mask = self._truncate(num_mask, max_length, truncation)

            if pad_to_max and max_length is not None and len(vals) < max_length:
                pad_len = max_length - len(vals)
                vals += [0.0] * pad_len
                num_mask += [0] * pad_len

            out["numeric_values"] = vals
            out["numeric_mask"] = num_mask

        # Text stream
        if self.return_text:
            if "text_value" in df.columns:
                txt = df.get_column("text_value").to_list()
            else:
                txt = [None] * df.height
            txt = [("" if (t is None or str(t) == "___") else str(t)) for t in txt]
            txt_mask = [1 if (t != "") else 0 for t in txt]

            txt = self._truncate(txt, max_length, truncation)
            txt_mask = self._truncate(txt_mask, max_length, truncation)

            if pad_to_max and max_length is not None and len(txt) < max_length:
                pad_len = max_length - len(txt)
                txt += [""] * pad_len
                txt_mask += [0] * pad_len

            out["text_values"] = txt
            out["text_mask"] = txt_mask

            
        if self.return_time:
            if "time_diff" in df.columns:
                df = df.with_columns(pl.col(['time_diff'])).fill_null(0.0)
                time_diff = df.get_column("time_diff").to_list()
                time_diff = self._scale_time_deltas(time_diff)
                time_stamp = df.get_column("time").to_list()
            else:
                time_diff = [None] * df.height
                time_stamp = [None] * df.height


            if pad_to_max and max_length is not None and len(time_diff) < max_length:
                pad_len = max_length - len(time_diff)
                time_diff += [0] * pad_len
                time_stamp += [0] * pad_len


            out["time_diff"] = time_diff
            out["time_stamp"] = time_stamp
            
        if self.return_ids:
            if "seq_id" in df.columns:
                
                seq_id = df.get_column("seq_id").cast(pl.Int32).to_list()
                out_id = df.get_column("out_id").cast(pl.Int32).to_list()
                er_id =  df.get_column("er_id").cast(pl.Int32).to_list()
                hadm_id = df.get_column("hadm_id").cast(pl.Int32).to_list()
                icustay_id = df.get_column("hadm_id").cast(pl.Int32).to_list()
            else:
                seq_id = [None] * df.height
                out_id = [None] * df.height
                er_id =  [None] * df.height
                hadm_id = [None] * df.height
                icustay_id = [None] * df.height

            if pad_to_max and max_length is not None and len(time_diff) < max_length:
                pad_len = max_length - len(time_diff)
                seq_id += [0] * pad_len
                out_id += [0] * pad_len
                er_id += [0] * pad_len
                hadm_id += [0] * pad_len
                icustay_id += [0] * pad_len

            out["seq_id"] = seq_id
            out["out_id"] = out_id
            out["er_id"] = er_id
            out["hadm_id"] = hadm_id
            out["icustay_id"] = icustay_id

        return out
    
    def _scale_time_deltas(self, deltas_list):
        deltas = np.asarray(deltas_list, dtype=float)
        compressed = np.log1p(deltas)              
        scaled = compressed / np.log(5328.93125)         
        return scaled.tolist()

    @staticmethod
    def _truncate(seq: List[Any], max_length: Optional[int], truncation: str) -> List[Any]:
        if max_length is None or len(seq) <= max_length:
            return seq
        return seq[-max_length:] if truncation == "head" else seq[:max_length]

    @staticmethod
    def _pad_list(lst: List[Any], pad_len: int, pad_value: Any) -> List[Any]:
        if pad_len <= 0:
            return lst
        return lst + [pad_value] * pad_len

In [5]:
# def get_adm_disch_time(data_path: str) -> pl.DataFrame:
    
#     pieces = []
#     for file in tqdm(os.listdir(data_path)):
#         # hospital admission data
#         data_hosp = (pl.scan_parquet(os.path.join(data_path,file))
#                   .select(['subject_id','hadm_id','time','icustay_id','code'])
#                   .collect())
        
#         hosp_adm = (data_hosp.filter(pl.col('code') == 'ADMISSION-AT-HOSPITAL')[:,:-2]
#                              .rename({'time':'hosp_admission_time'}))
#         hosp_disch = (data_hosp.filter(pl.col('code') == 'DISCHARGE-FROM-HOSPITAL')[:,:-2]
#                                .rename({'time':'hosp_discharge_time'}))

#         adm_disch = hosp_adm.join(hosp_disch, on=['subject_id','hadm_id'], how='left')
        
        
#         # ICU admission data
#         data_icu = data_hosp.filter(pl.col('icustay_id').is_null() == False)
#         icu_adm = (data_icu.filter(pl.col('code') == 'ADMISSION-AT-ICU')[:,:-1]
#                            .rename({'time':'icu_admission_time'}))
#         icu_disch = (data_icu.filter(pl.col('code') == 'DISCHARGE-FROM-ICU')[:,:-1]
#                              .rename({'time':'icu_discharge_time'}))
#         icu_adm_disch = icu_adm.join(icu_disch,on=['subject_id','hadm_id','icustay_id'], how='left')
#         icu_adm_disch = icu_adm_disch['subject_id','hadm_id','icustay_id','icu_admission_time','icu_discharge_time']
#         adm_disch = adm_disch.join(icu_adm_disch,on=['subject_id','hadm_id'],how='left')
        
        
#         # In-hospital motality data
#         hosp_mort = (data_hosp.filter(pl.col('code') == 'DISCHARGE-lOCATION//DIED')
#                               .sort('subject_id')
#                               .rename({'time':'in_hosp_mort_time'}))[:,:-2]
        
#         adm_disch = adm_disch.join(hosp_mort,on=['subject_id','hadm_id'],how='left')
        
#         # out-of-hospital motality data
#         out_mort = (data_hosp.filter(pl.col('code') == 'MEDS_DEATH')
#                              .rename({'time':'out_mortality_time'})
#                              .filter(~pl.col('subject_id')
#                              .is_in((hosp_mort['subject_id']).implode()))).drop(['hadm_id','code','icustay_id'])
        
#         # Count number of events in ICU and hospital
#         n_events_hosp = (data_hosp.group_by('subject_id','hadm_id')
#                                   .len()
#                                   .rename({'len':'n_events_hosp'}))
        
#         n_events_icu = (data_hosp.group_by('subject_id','icustay_id')
#                                   .len()
#                                   .rename({'len':'n_events_icu'}))
        
#         adm_disch = adm_disch.join(out_mort, on=['subject_id'], how='left')
#         adm_disch = adm_disch.join(n_events_hosp, on=['subject_id','hadm_id'], how='left')
#         adm_disch = adm_disch.join(n_events_icu, on=['subject_id','icustay_id'], how='left')
        
#         adm_disch = adm_disch.with_columns(pl.lit(file).alias('shard'))
        
#         pieces.append(adm_disch)
        
        
#     data = (pl.concat(pieces, how="vertical").sort('subject_id'))

    
#     data = (data.sort(["subject_id", "hosp_admission_time"])
#                  .with_columns(pl.when(
#                      pl.col("hosp_admission_time") != 
#                      pl.col("hosp_admission_time").max().over("subject_id"))
#                 .then(None)
#                 .otherwise(pl.col("out_mortality_time"))
#                 .alias("out_mortality_time")))
#     return data
        
        

In [6]:
# data_idx = get_adm_disch_time('../data/meds_normalized/data/train/')

In [7]:
# def get_length_of_stay(data_idx: pl.DataFrame) -> pl.DataFrame:
    
#     data_idx = data_idx.with_columns(((pl.col("hosp_discharge_time") - pl.col("hosp_admission_time"))
#                                          .alias("hosp_los")))
#     data_idx = data_idx.with_columns(((pl.col("hosp_discharge_time") - pl.col("hosp_admission_time"))
#                                          .alias("hosp_los_hours")
#                                          .dt.total_seconds())/3600)
#     data_idx = data_idx.with_columns(((pl.col("hosp_discharge_time") - pl.col("hosp_admission_time"))
#                                      .alias("hosp_los_days")
#                                      .dt.total_seconds())/(3600*24))
    
    
    
#     data_idx = data_idx.with_columns(((pl.col("icu_discharge_time") - pl.col("icu_admission_time"))
#                                          .alias("icu_los")))
#     data_idx = data_idx.with_columns(((pl.col("icu_discharge_time") - pl.col("icu_admission_time"))
#                                          .alias("icu_los_hours")
#                                          .dt.total_seconds())/3600)
#     data_idx = data_idx.with_columns(((pl.col("icu_discharge_time") - pl.col("icu_admission_time"))
#                                      .alias("icu_los_days")
#                                      .dt.total_seconds())/(3600*24))
    
    
#     return data_idx

In [8]:
# data_idx = get_length_of_stay(data_idx)

# Full dataset stats:

1. Number of admissions: 498,015

2. Num ICU Stays: 94,456

In [9]:
# # total patients extracted 
# num_patients = data_idx['subject_id'].unique().shape[0]
# print('Number of patients:', num_patients)

# print('\n')


# num_admissions = data_idx['hadm_id'].unique().shape[0]
# print('Number of hospital admissions:', num_admissions)

# print('\n')

# num_admissions_icu = data_idx['icustay_id'].unique().shape[0] - 1
# print('Number of icu stays:', num_admissions_icu)



# ICU dataset only
1. Cohort is based on those patient admitted to the ICU
2. TODO: Consider building separate dataset for hospital

In [10]:
# # get admissions with ICU stay only
# data_idx = data_idx.filter(pl.col('icustay_id').is_not_null())

In [11]:
# # exclude admissions with more than one (1) icu stay within the same admission 

# icustays_per_admission = data_idx.group_by(['subject_id','hadm_id']).len().rename({'len':'n_icu_stays'})

# icustays_per_admission = icustays_per_admission.filter(pl.col('n_icu_stays')==1)

# data_idx = data_idx.join(icustays_per_admission, on=['subject_id','hadm_id'], how='inner').drop('n_icu_stays')

In [12]:
# # Exclude patients with ICU los less than 24
# data_idx = data_idx.filter(pl.col('icu_los_hours') > 24)


In [13]:
# Exclude patients with ICU events less than 1024

# data_idx = data_idx.filter(pl.col('n_events_icu') > 1024)

In [14]:
# data_idx = data_idx.with_columns(pl.col('icu_admission_time').dt.offset_by("24h").alias("mort_24hr_offset"))
# data_idx = data_idx.with_columns(pl.col('icu_admission_time').dt.offset_by("48h").alias("mort_48hr_offset"))

In [15]:



# # In hospital mortality labels
# data_idx = data_idx.with_columns((pl.col("in_hosp_mort_time").is_not_null()).cast(pl.Int8).alias("y_mort"))
# data_idx = data_idx.with_columns((pl.col("out_mortality_time").is_not_null()).cast(pl.Int8).alias("y_mort_1yr"))



# # long length of stay labels
# data_idx = data_idx.with_columns((pl.col("icu_los_days") >= 7).cast(pl.Int8).alias("y_los_7"))
# data_idx = data_idx.with_columns((pl.col("icu_los_days") >= 15).cast(pl.Int8).alias("y_los_15"))
# data_idx = data_idx.with_columns((pl.col("icu_los_days") >= 30).cast(pl.Int8).alias("y_los_30"))

# # ICU readmission
# data_idx = (data_idx.sort(["subject_id", "icu_admission_time"]) 
#                     .with_columns(
#                     (pl.col("icu_admission_time").shift(-1).over("subject_id").is_not_null())
#                     .cast(pl.Int8).alias("y_icu_readmit")))

# data_idx = (
#     data_idx
#     .sort(["subject_id", "icu_admission_time"])
#     .with_columns(
#         # next ICU admission for the same subject
#         pl.col("icu_admission_time").shift(-1).over("subject_id").alias("next_icu_admit")
#     )
#     .with_columns(
#         # gap from this ICU discharge to the next ICU admission
#         (pl.col("next_icu_admit") - pl.col("icu_discharge_time")).alias("gap_to_next_icu")
#     )
#     .with_columns(
#         # convenience masks
#         icu_readmit_possible = pl.col("icu_discharge_time").is_not_null() & pl.col("next_icu_admit").is_not_null(),
#         icu_gap_valid = pl.col("gap_to_next_icu") > pl.duration(days=0),  # strictly after discharge
#     )
#     .with_columns(
#         y_icu_readmit_7  = (pl.col("icu_readmit_possible") & pl.col("icu_gap_valid") &
#                             (pl.col("gap_to_next_icu") <= pl.duration(days=7))).cast(pl.Int8),
#         y_icu_readmit_15 = (pl.col("icu_readmit_possible") & pl.col("icu_gap_valid") &
#                             (pl.col("gap_to_next_icu") <= pl.duration(days=15))).cast(pl.Int8),
#         y_icu_readmit_30 = (pl.col("icu_readmit_possible") & pl.col("icu_gap_valid") &
#                             (pl.col("gap_to_next_icu") <= pl.duration(days=30))).cast(pl.Int8),
#     )
#     .with_columns([
#         pl.col("y_icu_readmit_7").fill_null(0),
#         pl.col("y_icu_readmit_15").fill_null(0),
#         pl.col("y_icu_readmit_30").fill_null(0),
#     ])
#     .drop(["next_icu_admit", "gap_to_next_icu", "icu_readmit_possible", "icu_gap_valid"])
# )

In [16]:
# from sklearn.model_selection import train_test_split

In [17]:
# # total_patients = 49_839
# # total_stays = 61_175

# # train_patients = 34887# 80
# # val_patients =    4984 # 10
# # test_patients = 9968  # 20 

# SEED = 24

# train1, test = train_test_split(data_idx['subject_id'].unique(),train_size=0.8, random_state=SEED)
# train, val = train_test_split(train1,test_size=0.125, random_state=SEED)

In [18]:
# train_split = data_idx.filter(pl.col('subject_id').is_in(pl.lit(train).implode()))
# val_split = data_idx.filter(pl.col('subject_id').is_in(pl.lit(val).implode()))
# test_split = data_idx.filter(pl.col('subject_id').is_in(pl.lit(test).implode()))

In [19]:
# train_split = train_split.with_columns(pl.lit('train').alias('split'))
# val_split = val_split.with_columns(pl.lit('val').alias('split'))
# test_split = test_split.with_columns(pl.lit('test').alias('split'))

In [20]:
# print('in_hosp_mortality \n')
# task = 'y_mort'

# print('overall positive stays = ',data_idx.filter(pl.col(task) == 1).shape[0],' | P= ', 
#       data_idx.filter(pl.col(task) == 1).shape[0]/data_idx.shape[0])
# print('overall negative stays = ',data_idx.filter(pl.col(task) == 0).shape[0],' | P= ',
#       data_idx.filter(pl.col(task) == 0).shape[0]/data_idx.shape[0])
# print('\n')
# print('train positive stays = ',train_split.filter(pl.col(task) == 1).shape[0],' | P= ',
#      train_split.filter(pl.col(task) == 1).shape[0]/train_split.shape[0])
# print('train negative stays = ',train_split.filter(pl.col(task) == 0).shape[0],' | P= ',
#      train_split.filter(pl.col(task) == 0).shape[0]/train_split.shape[0])
# print('\n')
# print('val positive stays = ',val_split.filter(pl.col(task) == 1).shape[0],' | P= ',
#      val_split.filter(pl.col(task) == 1).shape[0]/val_split.shape[0])
# print('val negative stays = ',val_split.filter(pl.col(task) == 0).shape[0],' | P= ',
#      val_split.filter(pl.col(task) == 0).shape[0]/val_split.shape[0])
# print('\n')
# print('test positive stays = ',test_split.filter(pl.col(task) == 1).shape[0],' | P= ',
#      test_split.filter(pl.col(task) == 1).shape[0]/test_split.shape[0])
# print('test negative stays = ',test_split.filter(pl.col(task) == 0).shape[0],' | P= ',
#      test_split.filter(pl.col(task) == 0).shape[0]/test_split.shape[0])



# print('*'*50)
# print('1-year mortality \n')
# task = 'y_mort_1yr'
# print('overall positive stays = ',data_idx.filter(pl.col(task)==1).shape[0],' | P= ',
#       data_idx.filter(pl.col(task)==1).shape[0]/data_idx.shape[0])
# print('overall negative stays = ',data_idx.filter(pl.col(task)==0).shape[0],' | P= ',
#       data_idx.filter(pl.col(task)==0).shape[0]/data_idx.shape[0])
# print('\n')
# print('train positive stays = ',train_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       train_split.filter(pl.col(task)==1).shape[0]/train_split.shape[0])
# print('train negative stays = ',train_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       train_split.filter(pl.col(task)==0).shape[0]/train_split.shape[0])
# print('\n')
# print('val positive stays = ',val_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       val_split.filter(pl.col(task)==1).shape[0]/val_split.shape[0])
# print('val negative stays = ',val_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       val_split.filter(pl.col(task)==0).shape[0]/val_split.shape[0])
# print('\n')
# print('test positive stays = ',test_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       test_split.filter(pl.col(task)==1).shape[0]/test_split.shape[0])
# print('test negative stays = ',test_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       test_split.filter(pl.col(task)==0).shape[0]/test_split.shape[0])



# print('*'*50)
# print('long los 7 \n')
# task = 'y_los_7'

# print('overall positive stays = ',data_idx.filter(pl.col(task)==1).shape[0],' | P= ',
#       data_idx.filter(pl.col(task)==1).shape[0]/data_idx.shape[0])
# print('overall negative stays = ',data_idx.filter(pl.col(task)==0).shape[0],' | P= ',
#       data_idx.filter(pl.col(task)==0).shape[0]/data_idx.shape[0])
# print('\n')
# print('train positive stays = ',train_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       train_split.filter(pl.col(task)==1).shape[0]/train_split.shape[0])
# print('train negative stays = ',train_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       train_split.filter(pl.col(task)==0).shape[0]/train_split.shape[0])
# print('\n')
# print('val positive stays = ',val_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       val_split.filter(pl.col(task)==1).shape[0]/val_split.shape[0])
# print('val negative stays = ',val_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       val_split.filter(pl.col(task)==0).shape[0]/val_split.shape[0])
# print('\n')
# print('test positive stays = ',test_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       test_split.filter(pl.col(task)==1).shape[0]/test_split.shape[0])
# print('test negative stays = ',test_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       test_split.filter(pl.col(task)==0).shape[0]/test_split.shape[0])



# print('*'*50)
# print('long los 15 \n')
# task = 'y_los_15'

# print('overall positive stays = ',data_idx.filter(pl.col(task)==1).shape[0],' | P= ',
#       data_idx.filter(pl.col(task)==1).shape[0]/data_idx.shape[0])
# print('overall negative stays = ',data_idx.filter(pl.col(task)==0).shape[0],' | P= ',
#       data_idx.filter(pl.col(task)==0).shape[0]/data_idx.shape[0])
# print('\n')
# print('train positive stays = ',train_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       train_split.filter(pl.col(task)==1).shape[0]/train_split.shape[0])
# print('train negative stays = ',train_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       train_split.filter(pl.col(task)==0).shape[0]/train_split.shape[0])
# print('\n')
# print('val positive stays = ',val_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       val_split.filter(pl.col(task)==1).shape[0]/val_split.shape[0])
# print('val negative stays = ',val_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       val_split.filter(pl.col(task)==0).shape[0]/val_split.shape[0])
# print('\n')
# print('test positive stays = ',test_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       test_split.filter(pl.col(task)==1).shape[0]/test_split.shape[0])
# print('test negative stays = ',test_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       test_split.filter(pl.col(task)==0).shape[0]/test_split.shape[0])


# print('*'*50)
# print('long los 30 \n')
# task = 'y_los_30'

# print('overall positive stays = ',data_idx.filter(pl.col(task)==1).shape[0],' | P= ',
#       data_idx.filter(pl.col(task)==1).shape[0]/data_idx.shape[0])
# print('overall negative stays = ',data_idx.filter(pl.col(task)==0).shape[0],' | P= ',
#       data_idx.filter(pl.col(task)==0).shape[0]/data_idx.shape[0])
# print('\n')
# print('train positive stays = ',train_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       train_split.filter(pl.col(task)==1).shape[0]/train_split.shape[0])
# print('train negative stays = ',train_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       train_split.filter(pl.col(task)==0).shape[0]/train_split.shape[0])
# print('\n')
# print('val positive stays = ',val_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       val_split.filter(pl.col(task)==1).shape[0]/val_split.shape[0])
# print('val negative stays = ',val_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       val_split.filter(pl.col(task)==0).shape[0]/val_split.shape[0])
# print('\n')
# print('test positive stays = ',test_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       test_split.filter(pl.col(task)==1).shape[0]/test_split.shape[0])
# print('test negative stays = ',test_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       test_split.filter(pl.col(task)==0).shape[0]/test_split.shape[0])

# print('*'*50)
# print('ICU readmission general \n')
# task = 'y_icu_readmit'

# print('overall positive stays = ',data_idx.filter(pl.col(task)==1).shape[0],' | P= ',
#       data_idx.filter(pl.col(task)==1).shape[0]/data_idx.shape[0])
# print('overall negative stays = ',data_idx.filter(pl.col(task)==0).shape[0],' | P= ',
#       data_idx.filter(pl.col(task)==0).shape[0]/data_idx.shape[0])
# print('\n')
# print('train positive stays = ',train_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       train_split.filter(pl.col(task)==1).shape[0]/train_split.shape[0])
# print('train negative stays = ',train_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       train_split.filter(pl.col(task)==0).shape[0]/train_split.shape[0])
# print('\n')
# print('val positive stays = ',val_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       val_split.filter(pl.col(task)==1).shape[0]/val_split.shape[0])
# print('val negative stays = ',val_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       val_split.filter(pl.col(task)==0).shape[0]/val_split.shape[0])
# print('\n')
# print('test positive stays = ',test_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       test_split.filter(pl.col(task)==1).shape[0]/test_split.shape[0])
# print('test negative stays = ',test_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       test_split.filter(pl.col(task)==0).shape[0]/test_split.shape[0])

# print('*'*50)
# print('ICU readmission 7\n')
# task = 'y_icu_readmit_7'

# print('overall positive stays = ',data_idx.filter(pl.col(task)==1).shape[0],' | P= ',
#       data_idx.filter(pl.col(task)==1).shape[0]/data_idx.shape[0])
# print('overall negative stays = ',data_idx.filter(pl.col(task)==0).shape[0],' | P= ',
#       data_idx.filter(pl.col(task)==0).shape[0]/data_idx.shape[0])
# print('\n')
# print('train positive stays = ',train_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       train_split.filter(pl.col(task)==1).shape[0]/train_split.shape[0])
# print('train negative stays = ',train_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       train_split.filter(pl.col(task)==0).shape[0]/train_split.shape[0])
# print('\n')
# print('val positive stays = ',val_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       val_split.filter(pl.col(task)==1).shape[0]/val_split.shape[0])
# print('val negative stays = ',val_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       val_split.filter(pl.col(task)==0).shape[0]/val_split.shape[0])
# print('\n')
# print('test positive stays = ',test_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       test_split.filter(pl.col(task)==1).shape[0]/test_split.shape[0])
# print('test negative stays = ',test_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       test_split.filter(pl.col(task)==0).shape[0]/test_split.shape[0])


# print('*'*50)
# print('ICU readmission 15\n')
# task = 'y_icu_readmit_15'

# print('overall positive stays = ',data_idx.filter(pl.col(task)==1).shape[0],' | P= ',
#       data_idx.filter(pl.col(task)==1).shape[0]/data_idx.shape[0])
# print('overall negative stays = ',data_idx.filter(pl.col(task)==0).shape[0],' | P= ',
#       data_idx.filter(pl.col(task)==0).shape[0]/data_idx.shape[0])
# print('\n')
# print('train positive stays = ',train_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       train_split.filter(pl.col(task)==1).shape[0]/train_split.shape[0])
# print('train negative stays = ',train_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       train_split.filter(pl.col(task)==0).shape[0]/train_split.shape[0])
# print('\n')
# print('val positive stays = ',val_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       val_split.filter(pl.col(task)==1).shape[0]/val_split.shape[0])
# print('val negative stays = ',val_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       val_split.filter(pl.col(task)==0).shape[0]/val_split.shape[0])
# print('\n')
# print('test positive stays = ',test_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       test_split.filter(pl.col(task)==1).shape[0]/test_split.shape[0])
# print('test negative stays = ',test_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       test_split.filter(pl.col(task)==0).shape[0]/test_split.shape[0])


# print('*' * 50)
# print('ICU readmission 30\n')
# task = 'y_icu_readmit_30'

# print('overall positive stays = ',data_idx.filter(pl.col(task)==1).shape[0],' | P= ',
#       data_idx.filter(pl.col(task)==1).shape[0]/data_idx.shape[0])
# print('overall negative stays = ',data_idx.filter(pl.col(task)==0).shape[0],' | P= ',
#       data_idx.filter(pl.col(task)==0).shape[0]/data_idx.shape[0])
# print('\n')
# print('train positive stays = ',train_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       train_split.filter(pl.col(task)==1).shape[0]/train_split.shape[0])
# print('train negative stays = ',train_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       train_split.filter(pl.col(task)==0).shape[0]/train_split.shape[0])
# print('\n')
# print('val positive stays = ',val_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       val_split.filter(pl.col(task)==1).shape[0]/val_split.shape[0])
# print('val negative stays = ',val_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       val_split.filter(pl.col(task)==0).shape[0]/val_split.shape[0])
# print('\n')
# print('test positive stays = ',test_split.filter(pl.col(task)==1).shape[0],' | P= ',
#       test_split.filter(pl.col(task)==1).shape[0]/test_split.shape[0])
# print('test negative stays = ',test_split.filter(pl.col(task)==0).shape[0],' | P= ',
#       test_split.filter(pl.col(task)==0).shape[0]/test_split.shape[0])

In [21]:
# downstream_idx = pl.concat((train_split,val_split,test_split),how='vertical').sort('subject_id')

In [22]:
# pieces = []
# for i in tqdm(range(len(downstream_idx))):
    
#     shard = downstream_idx[i]['shard'][0]
#     path = f'../data/meds_normalized/data/train/{shard}'
#     cols = ['subject_id', 'seq_id','icustay_id','time', 'code','numeric_value','text_value']
#     subject_id = downstream_idx[i]['subject_id'][0]
#     hadm_id = downstream_idx[i]['hadm_id'][0]
#     icustay_id = downstream_idx[i]['icustay_id'][0]
#     icu_admission_time = downstream_idx[i]['icu_admission_time'][0]
#     icu_discharge_time = downstream_idx[i]['icu_discharge_time'][0]
#     mort_24hr_offset = downstream_idx[i]['mort_24hr_offset'][0]
#     mort_48hr_offset = downstream_idx[i]['mort_48hr_offset'][0]

#     df = (
#         pl.scan_parquet(path)
#         .select(cols)
#         .collect()
#         .filter(pl.col("subject_id") == subject_id)
#         .with_row_index(name="event_index")
#         .with_columns(
#             (
#                 pl.col("icustay_id").eq(icustay_id)
#                 & pl.col("time").ge(pl.lit(icu_admission_time))
#                 & pl.col("time").le(pl.lit(icu_discharge_time))
#             ).alias("icu_mask"),

#             # 24h window within the ICU stay: [icu_admission_time, mort_24hr_offset]
#             (
#                 pl.col("icustay_id").eq(icustay_id)
#                 & pl.col("time").ge(pl.lit(icu_admission_time))
#                 & pl.col("time").le(pl.lit(mort_24hr_offset))
#             ).alias("within_24h_mask"),
#             # 48h window within the ICU stay: [icu_admission_time, mort_48hr_offset]
#             (
#                 pl.col("icustay_id").eq(icustay_id)
#                 & pl.col("time").ge(pl.lit(icu_admission_time))
#                 & pl.col("time").le(pl.lit(mort_48hr_offset))
#             ).alias("within_48h_mask"),
#         )
#     )


#     df = df.with_columns(pl.col("event_index").cast(pl.Int64))
#     bounds = df.select(

#         pl.lit(subject_id).cast(pl.Int64).alias("subject_id"),
#         pl.lit(hadm_id).cast(pl.Int64).alias("hadm_id"),
#         pl.lit(icustay_id).cast(pl.Int64).alias("icustay_id"),
#         # 24h
#         pl.when(pl.col("within_24h_mask")).then(pl.col("event_index")).otherwise(None).min().alias("w24_min"),
#         pl.when(pl.col("within_24h_mask")).then(pl.col("event_index")).otherwise(None).max().alias("w24_max"),
#         # 48h
#         pl.when(pl.col("within_48h_mask")).then(pl.col("event_index")).otherwise(None).min().alias("w48_min"),
#         pl.when(pl.col("within_48h_mask")).then(pl.col("event_index")).otherwise(None).max().alias("w48_max"),
#         # ENTIRE STAY
#         pl.when(pl.col("icu_mask")).then(pl.col("event_index")).otherwise(None).min().alias("wStay_min"),
#         pl.when(pl.col("icu_mask")).then(pl.col("event_index")).otherwise(None).max().alias("wStay_max"),
#     )

#     # derive start/end for last (L-1) tokens in each window (L in {512,1024,1536} → k in {511,1023,1535})
#     bounds = bounds.with_columns(
#         # --- 24h ---
#         pl.max_horizontal(pl.col("w24_min"),  pl.col("w24_max")  - pl.lit(511)).alias("w24_start_512"),
#         pl.col("w24_max").alias("w24_end_512"),
#         pl.max_horizontal(pl.col("w24_min"),  pl.col("w24_max")  - pl.lit(1023)).alias("w24_start_1024"),
#         pl.col("w24_max").alias("w24_end_1024"),
#         pl.max_horizontal(pl.col("w24_min"),  pl.col("w24_max")  - pl.lit(1535)).alias("w24_start_1536"),
#         pl.col("w24_max").alias("w24_end_1536"),

#         # --- 48h ---
#         pl.max_horizontal(pl.col("w48_min"),  pl.col("w48_max")  - pl.lit(511)).alias("w48_start_512"),
#         pl.col("w48_max").alias("w48_end_512"),
#         pl.max_horizontal(pl.col("w48_min"),  pl.col("w48_max")  - pl.lit(1023)).alias("w48_start_1024"),
#         pl.col("w48_max").alias("w48_end_1024"),
#         pl.max_horizontal(pl.col("w48_min"),  pl.col("w48_max")  - pl.lit(1535)).alias("w48_start_1536"),
#         pl.col("w48_max").alias("w48_end_1536"),

#         # --- ENTIRE STAY ---
#         pl.max_horizontal(pl.col("wStay_min"), pl.col("wStay_max") - pl.lit(511)).alias("wStay_start_512"),
#         pl.col("wStay_max").alias("wStay_end_512"),
#         pl.max_horizontal(pl.col("wStay_min"), pl.col("wStay_max") - pl.lit(1023)).alias("wStay_start_1024"),
#         pl.col("wStay_max").alias("wStay_end_1024"),
#         pl.max_horizontal(pl.col("wStay_min"), pl.col("wStay_max") - pl.lit(1535)).alias("wStay_start_1536"),
#         pl.col("wStay_max").alias("wStay_end_1536"),
#     )
#     pieces.append(bounds)

# downstream_slices = pl.concat(pieces, how='vertical')

In [23]:
# downstream_slices.write_parquet('./downstream_slices.parquet')
# downstream_idx.write_parquet('./downstream_idx.parquet')
# pretrain_idx = pretrain_idx.filter(pl.col('subject_id').is_in(test_split['subject_id'].unique().implode()) == False)
# pretrain_idx.write_parquet('../pretrain_idx.parquet')
# pl.read_parquet('../pretrain_idx.parquet')
# pl.read_parquet('../downstream_idx.parquet')

In [24]:
# downstream_idx = downstream_idx.with_columns(pl.col(["hadm_id","icustay_id"]).cast(pl.Int64))
# downstream_idx = downstream_idx.join(downstream_slices, on=['subject_id','hadm_id','icustay_id'], how='inner')

In [25]:
# downstream_idx.write_parquet('./downstream_idx.parquet')

In [26]:
from datasets import load_from_disk
from collections import defaultdict

In [27]:
limits = {
    'within24_query': {512:  ['w24_start_512',  'w24_end_512' ],
                       1024: ['w24_start_1024', 'w24_end_1024'],
                       1536: ['w24_start_1536', 'w24_end_1536'],
                      },
    
    'within24_hist_icu': {512: ['wStay_min', 'w24_start_512' ],
                         1024: ['wStay_min', 'w24_start_1024'],
                         1536: ['wStay_min', 'w24_start_1536'],
                      },
    
    'within24_hist_full': {512: [ 0, 'w24_start_512' ],
                          1024: [ 0, 'w24_start_1024'],
                          1536: [ 0, 'w24_start_1536'],
                          },

    
    'within48_query': {512:  ['w48_start_512',  'w48_end_512' ],
                       1024: ['w48_start_1024', 'w48_end_1024'],
                       1536: ['w48_start_1536', 'w48_end_1536'],
                      },

    'within48_hist_icu': {512: ['wStay_min', 'w48_start_512' ],
                         1024: ['wStay_min', 'w48_start_1024'],
                         1536: ['wStay_min', 'w48_start_1536']
                      },
    
    'within48_hist_full': {512: [ 0, 'w48_start_512' ],
                          1024: [ 0, 'w48_start_1024'],
                          1536: [ 0, 'w48_start_1536']
                          },
    
    'within_stay_query': {512:  ['wStay_start_512',  'wStay_end_512' ],
                          1024: ['wStay_start_1024', 'wStay_end_1024'],
                          1536: ['wStay_start_1536', 'wStay_end_1536']
                         },
    
    'within_stay_hist_icu': {512:  ['wStay_min', 'wStay_start_512' ],
                             1024: ['wStay_min', 'wStay_start_1024'],
                             1536: ['wStay_min', 'wStay_start_1536'],
                            },
    
    'within_stay_hist_full': {512:  [ 0, 'w48_start_512' ],
                              1024: [ 0, 'w48_start_1024'],
                              1536: [ 0, 'w48_start_1536'],
                             },
    }

In [28]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
class EvalDataset(Dataset):
    def __init__(self,
                 dataset_path: str,
                 data_idx_path: str,
                 seq_gen: SequencesGenerator,
                 limits_dict: dict,
                 task: str = 'y_mort',
                 main_window: str = 'within48_query', 
                 seq_length: int = 512,
                 use_time: bool = True,
                 use_numeric: bool = False,
                 split: str = 'train') -> None:
        
        needed_cols = ['subject_id', 'input_ids', 'attention_mask', 
                       'visit_ids', 'stage_ids', 'type_ids']

        if use_time:
            needed_cols.append('time_diff')
        if use_numeric:
            needed_cols.append('numeric_values')
            needed_cols.append('numeric_mask')
        self.start_limit = limits_dict[main_window][seq_length][0]
        self.end_limit   = limits_dict[main_window][seq_length][1]
        self.task = task
        
        self.seq_gen = seq_gen
        self.data_idx =  pl.scan_parquet(data_idx_path).collect()
        self.data_idx =  self.data_idx.filter(pl.col('split') == split)
        
        sub_ids = set(self.data_idx.get_column("subject_id").to_list())
        hf_dataset = load_from_disk(dataset_path)

        
        hf_dataset = hf_dataset.filter(
            lambda sids: [sid in sub_ids for sid in sids],
            batched=True,
            input_columns="subject_id",
        )

        # (then continue)
        self.hf_dataset = (
            hf_dataset
            .flatten_indices()
            .select_columns(needed_cols)
            .with_format("numpy", columns=needed_cols, output_all_columns=False)
        )
        
        
        sids = self.hf_dataset["subject_id"]        
        self.index = defaultdict(list)
        for i, sid in enumerate(sids):
            self.index[sid].append(i)
        

        
    def __len__(self) -> int:
        return len(self.data_idx)


    
    def __getitem__(self,
                    idx: int):
        
        stay = self.data_idx[idx]
        subject_id = stay['subject_id'][0]
        label = stay[self.task][0]
       
        
        
        start = stay[self.start_limit][0]
        end = stay[self.end_limit][0]

        
        timeline_encoded = self.hf_dataset.select(self.index[subject_id])[0]
        prediction_window = {k: (v[start:end] if isinstance(v, (list, np.ndarray)) else v) for k, v in timeline_encoded.items()}
        prediction_window = self.seq_gen.get_overlapped_chunks(prediction_window)
        prediction_window[0]['label'] = label
        
        return prediction_window[0]

In [29]:
class EvalCollator:
    def __init__(self) -> None:
        pass

    def __call__(self, batch: List[Union[Dict, List[Dict]]]) -> Dict[str, torch.Tensor]:
        chunks = self._flatten(batch)
        out = self._stack(chunks)

        # ---- CLEAN NUMERIC VALUES ----
        if "numeric_values" in out:
            vals = out["numeric_values"].float()          # [B, L]
            finite_mask = torch.isfinite(vals)            # True where not NaN/inf

            # if numeric_mask already exists, AND it with finite_mask
            if "numeric_mask" in out:
                mask = out["numeric_mask"].bool() & finite_mask
            else:
                mask = finite_mask

            # replace NaN/inf with 0.0 (or any neutral value)
            vals = torch.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)

            out["numeric_values"] = vals
            out["numeric_mask"] = mask

#         # ---- OPTIONAL: CLEAN TIME FEATURES TOO ----
#         if "time_diff" in out:
#             t = out["time_diff"].float()
#             t = torch.nan_to_num(t, nan=0.0, posinf=0.0, neginf=0.0)
#             out["time_diff"] = t

        return out

    def _flatten(self, batch) -> List[Dict]:
        out: List[Dict] = []
        for item in batch:
            if isinstance(item, dict):
                out.append(item)
            elif isinstance(item, (list, tuple)):
                out.extend(item)
            else:
                raise TypeError(f"Unexpected item type: {type(item)}")
        if not out:
            raise ValueError("Empty batch after normalization.")
        return out

    def _stack(self, chunks: List[Dict]) -> Dict[str, torch.Tensor]:
        keys = list(chunks[0].keys())
        out = {}
        for k in keys:
            # Skip known non-numeric or variable-shaped fields
            if k in ("text_values",):  # add others you don't want to collate
                continue

            seq_list = []
            for c in chunks:
                v = c[k]
                if isinstance(v, list):
                    v = [0 if x is None else x for x in v]
                elif v is None:
                    v = 0
                seq_list.append(torch.as_tensor(v))
            out[k] = torch.stack(seq_list, 0)
        return out

In [30]:
from datasets import load_from_disk
dt = load_from_disk('./ehr_arrow_dataset/')
dt 

Loading dataset from disk:   0%|          | 0/70 [00:00<?, ?it/s]

Dataset({
    features: ['subject_id', 'input_ids', 'attention_mask', 'visit_ids', 'stage_ids', 'type_ids', 'numeric_values', 'numeric_mask', 'text_values', 'text_mask', 'time_diff', 'time_stamp', 'seq_id', 'out_id', 'er_id', 'hadm_id', 'icustay_id'],
    num_rows: 208980
})

In [31]:
seq_gen = SequencesGenerator(tokenizer_path='../vocab.json',
                             chunk_length=512,
                             overlap=0,
                              ) 
        
# collate_fn = EvalCollator()


train_dataset = EvalDataset(dataset_path='./ehr_arrow_dataset/',
                    data_idx_path='../downstream_idx.parquet',
                    seq_gen=seq_gen,
                    seq_length=512,
                    limits_dict=limits,
                    main_window='within48_query',
                    task='y_mort',
                    split='val')

Loading dataset from disk:   0%|          | 0/70 [00:00<?, ?it/s]

In [32]:
train_dataset.hf_dataset

Dataset({
    features: ['subject_id', 'input_ids', 'attention_mask', 'visit_ids', 'stage_ids', 'type_ids', 'time_diff'],
    num_rows: 4984
})

In [33]:
import torch
from torch import nn
from typing import Callable


class Time2Vec(nn.Module):

    def __init__(
        self,
        in_features: int = 1,
        out_features: int = 16,
        periodic_activation: Callable = torch.sin,
    ):
        super().__init__()
        assert out_features >= 1, "out_features must be >= 1"

        self.in_features = in_features
        self.out_features = out_features
        self.periodic_activation = periodic_activation

        self.W = nn.Parameter(torch.randn(in_features, out_features - 1))
        self.b = nn.Parameter(torch.randn(out_features - 1))

        self.W0 = nn.Parameter(torch.randn(in_features))
        self.b0 = nn.Parameter(torch.randn(1))

    def forward(self, tau: torch.Tensor) -> torch.Tensor:

        v1 = self.periodic_activation(tau @ self.W + self.b)
        v2 = (tau @ self.W0).unsqueeze(-1) + self.b0

        return torch.cat([v2, v1], dim=-1)


In [34]:
class EHREmbeddings(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_size: int,
        pad_token_id: int = 0,
        type_vocab_size: int = 28,
        visit_vocab_size: int = 102,
        stage_vocab_size: int = 5,
        dropout: float = 0.1,
        use_position_embeddings: bool = False,
        max_position_embeddings: int = 0,
        use_time: bool = True,
        time_in_features: int = 1,
        time_out_features: int = 16,
        use_numeric: bool = True,
        numeric_hidden_size: int = 16,   # <-- small bottleneck for numeric
    ):
        super().__init__()

        self.tok_emb   = nn.Embedding(vocab_size,       embedding_size, padding_idx=pad_token_id)
        self.type_emb  = nn.Embedding(type_vocab_size,  embedding_size, padding_idx=pad_token_id)
        self.visit_emb = nn.Embedding(visit_vocab_size, embedding_size, padding_idx=pad_token_id)
        self.stage_emb = nn.Embedding(stage_vocab_size, embedding_size, padding_idx=pad_token_id)

        # ---- positional (local) ----
        self.use_position_embeddings = use_position_embeddings
        if use_position_embeddings:
            if max_position_embeddings <= 0:
                raise ValueError("max_position_embeddings must be > 0 when use_position_embeddings=True")
            self.pos_emb = nn.Embedding(max_position_embeddings, embedding_size)
        else:
            self.pos_emb = None

        # ---- time (Time2Vec) ----
        self.use_time = use_time
        if use_time:
            self.time2vec = Time2Vec(
                in_features=time_in_features,
                out_features=time_out_features,
                periodic_activation=torch.sin,
            )
            self.time_proj = nn.Linear(time_out_features, embedding_size)
        else:
            self.time2vec = None
            self.time_proj = None

        # ---- numeric values ----
        self.use_numeric = use_numeric
        if use_numeric:
            self.numeric_hidden_size = numeric_hidden_size
            # 1 scalar -> small hidden -> embedding_size
            self.num_proj1 = nn.Linear(1, numeric_hidden_size)
            self.num_proj2 = nn.Linear(numeric_hidden_size, embedding_size)
            self.num_act = nn.GELU()

            # learned embedding for "no numeric value"
            self.null_numeric = nn.Parameter(torch.zeros(embedding_size))
            nn.init.normal_(self.null_numeric, mean=0.0, std=0.02)

            nn.init.xavier_uniform_(self.num_proj1.weight)
            nn.init.zeros_(self.num_proj1.bias)
            nn.init.xavier_uniform_(self.num_proj2.weight)
            nn.init.zeros_(self.num_proj2.bias)
        else:
            self.num_proj1 = None
            self.num_proj2 = None
            self.num_act = None
            self.null_numeric = None

        self.norm = nn.LayerNorm(embedding_size)
        self.drop = nn.Dropout(dropout)

    def encode(
        self,
        input_ids,
        type_ids,
        visit_ids,
        stage_ids,
        time_feats=None,          # (B, L) or (B, L, time_in_features)
        numeric_values=None,      # (B, L) normalized in [-3, 3]
        numeric_mask=None,        # (B, L) bool/int: True if numeric is present
    ):
        # base token + type + visit + stage
        x = self.tok_emb(input_ids.long())
        x = x + self.type_emb(type_ids.long())
        x = x + self.visit_emb(visit_ids.long())
        x = x + self.stage_emb(stage_ids.long())

        # positional (local) embeddings
        if self.pos_emb is not None:
            bsz, seqlen = input_ids.size()
            position_ids = torch.arange(
                seqlen, device=input_ids.device
            ).unsqueeze(0).expand(bsz, seqlen)
            x = x + self.pos_emb(position_ids)

        # time (Time2Vec)
        if self.use_time:
            if time_feats is None:
                raise ValueError("time_feats must be provided when use_time=True")
            if time_feats.dim() == 2:
                time_feats = time_feats.unsqueeze(-1)
            elif time_feats.dim() != 3:
                raise ValueError(f"Unexpected time_feats.dim()={time_feats.dim()}, expected 2 or 3")
            t = self.time2vec(time_feats.float())   # (B, L, time_out_features)
            t = self.time_proj(t)                   # (B, L, embedding_size)
            x = x + t

        # numeric values
        if self.use_numeric:
            if numeric_values is None or numeric_mask is None:
                raise ValueError("numeric_values and numeric_mask must be provided when use_numeric=True")

            # (optional safety) clamp extreme values
            v = numeric_values.float().unsqueeze(-1)        # (B, L, 1)
            # small bottleneck then project to emb size
            h = self.num_act(self.num_proj1(v))             # (B, L, H_num)
            num_emb = self.num_proj2(h)                     # (B, L, D)

            mask = numeric_mask.bool().unsqueeze(-1)        # (B, L, 1)
            num_emb = torch.where(mask, num_emb, self.null_numeric.view(1, 1, -1))
            x = x + num_emb

        return self.drop(self.norm(x))

    def forward(self, input_ids=None, token_type_ids=None, inputs_embeds=None, **kwargs):
        if inputs_embeds is not None:
            return inputs_embeds
        x = self.tok_emb(input_ids.long())
        return self.drop(self.norm(x))

In [35]:
from transformers  import RoFormerConfig, RoFormerModel
from torchmetrics.classification import BinaryAccuracy, BinaryAUROC, BinaryAveragePrecision
import lightning as lt

import torch
import torch.nn as nn

from transformers  import RoFormerConfig, RoFormerModel
from torchmetrics.classification import BinaryAccuracy, BinaryAUROC, BinaryAveragePrecision
import lightning as lt

import torch
import torch.nn as nn

class EvalModel(lt.LightningModule):
    def __init__(
        self,
        config,
        ckpt_path: str = None,
        lr: float = 2e-5,
        wd: float = 0.001,
        max_epochs: int = 100,
        dropout: float = 0.1,
        freeze: bool = False,
        pooling: str = 'cls',
        use_numeric: bool = True,
        use_time: bool = True 
    ):
        super().__init__()
        self.save_hyperparameters()
        self.pooling = pooling
        self.backbone = RoFormerModel(config)

        rope_model_types = {"modernbert", "roformer"}
        model_type = getattr(config, "model_type", "").lower()
        is_rope = model_type in rope_model_types

        self.train_step_preds = []
        self.train_step_label = []
        self.val_step_preds = []
        self.val_step_label = []
        self.test_step_preds = []
        self.test_step_label = []

        ehr_emb = EHREmbeddings(
            vocab_size=config.vocab_size,
            embedding_size=config.embedding_size,
            pad_token_id=config.pad_token_id,
            type_vocab_size=config.type_vocab_size,
            visit_vocab_size=config.visit_vocab_size,
            stage_vocab_size=config.stage_vocab_size,
            dropout=dropout,
            use_position_embeddings=not is_rope,
            max_position_embeddings=(
                getattr(config, "max_position_embeddings", 0)
                if not is_rope
                else 0
            ),
            use_time=use_time,
            time_in_features=1,
            time_out_features=16,
            use_numeric=use_numeric,  
        )

        self.backbone.embeddings = ehr_emb

        self.classifier = nn.Linear(config.hidden_size, 1)
        self.criterion = nn.BCEWithLogitsLoss()

        if ckpt_path:
            self.get_pretrained_weights(model=self.backbone, ckpt_path=ckpt_path)

        if freeze:
            for param in self.backbone.parameters():
                param.requires_grad = False
            for param in self.classifier.parameters():
                param.requires_grad = True

        self.lr = lr
        self.wd = wd
        self.max_epochs = max_epochs

        # metrics (unchanged)
        self.train_auroc = BinaryAUROC()
        self.train_auprc = BinaryAveragePrecision()
        self.val_auroc = BinaryAUROC()
        self.val_auprc = BinaryAveragePrecision()
        self.test_auroc = BinaryAUROC()
        self.test_auprc = BinaryAveragePrecision()

    def forward(
        self,
        input_ids,
        attention_mask,
        type_ids,
        visit_ids,
        stage_ids,
        time_feats=None,
        numeric_values=None,  
        numeric_mask=None,    
        labels=None,
    ):
        inputs_embeds = self.backbone.embeddings.encode(
            input_ids=input_ids,
            type_ids=type_ids,
            visit_ids=visit_ids,
            stage_ids=stage_ids,
            time_feats=time_feats,
            numeric_values=numeric_values,
            numeric_mask=numeric_mask,      
        )

        outputs = self.backbone(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            output_hidden_states=False,
            return_dict=True,
        )
        last_hidden = outputs.last_hidden_state  

        # pooling
        if self.pooling == 'mean':
            mask = attention_mask.unsqueeze(-1).type_as(last_hidden)  
            summed = (last_hidden * mask).sum(dim=1)                 
            lengths = mask.sum(dim=1).clamp(min=1.0)                  
            pooled = summed / lengths                               
        elif self.pooling == 'cls':
            pooled = last_hidden[:, 0, :]

        logits = self.classifier(pooled).squeeze(-1)              
        return logits

    def training_step(self, batch, batch_idx):

        logits = self.forward(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            type_ids=batch["type_ids"],
            visit_ids=batch["visit_ids"],
            stage_ids=batch["stage_ids"],
            time_feats=batch.get("time_diff", None),
            numeric_values=batch.get("numeric_values", None),  
            numeric_mask=batch.get("numeric_mask", None),    
            labels=None,
        )

        y = batch["label"].float().view(-1)    
        loss = self.criterion(logits, y)

        pos_score = torch.sigmoid(logits)       

        self.train_step_label.append(y)
        self.train_step_preds.append(pos_score)

        self.log("train_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_train_epoch_end(self) -> None:
        y = torch.cat(self.train_step_label)
        pos_score = torch.cat(self.train_step_preds)

        auroc = self.train_auroc(pos_score, y.long())
        auprc = self.train_auprc(pos_score, y.long())

        self.log('train_auroc', auroc, on_epoch=True, logger=True, prog_bar=True)
        self.log('train_auprc', auprc, on_epoch=True, logger=True, prog_bar=True)

        self.train_step_label.clear()
        self.train_step_preds.clear()

    def validation_step(self, batch, batch_idx):
        logits = self.forward(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            type_ids=batch["type_ids"],
            visit_ids=batch["visit_ids"],
            stage_ids=batch["stage_ids"],
            time_feats=batch.get("time_diff", None),
            numeric_values=batch.get("numeric_values", None),  
            numeric_mask=batch.get("numeric_mask", None),    
            labels=None,
        )

        y = batch["label"].float().view(-1)
        loss = self.criterion(logits, y)
        pos_score = torch.sigmoid(logits)

        self.val_step_label.append(y)
        self.val_step_preds.append(pos_score)

        self.log("val_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_validation_epoch_end(self,*arg, **kwargs) -> None:
        y = torch.cat(self.val_step_label)
        pos_score = torch.cat(self.val_step_preds)

        auroc = self.val_auroc(pos_score, y.long())
        auprc = self.val_auprc(pos_score, y.long())

        self.log('val_auroc', auroc, on_epoch=True, logger=True, prog_bar=True)
        self.log('val_auprc', auprc, on_epoch=True, logger=True, prog_bar=True)

        self.val_step_label.clear()
        self.val_step_preds.clear()

    def test_step(self, batch, batch_idx):
        logits = self.forward(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            type_ids=batch["type_ids"],
            visit_ids=batch["visit_ids"],
            stage_ids=batch["stage_ids"],
            time_feats=batch.get("time_diff", None),
            numeric_values=batch.get("numeric_values", None), 
            numeric_mask=batch.get("numeric_mask", None),     
            labels=None,
        )

        y = batch["label"].float().view(-1)
        loss = self.criterion(logits, y)
        pos_score = torch.sigmoid(logits)

        self.test_step_label.append(y)
        self.test_step_preds.append(pos_score)

        self.log("test_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss    

    def on_test_epoch_end(self,*arg, **kwargs) -> None:
        y = torch.cat(self.test_step_label)
        pos_score = torch.cat(self.test_step_preds)

        auroc = self.test_auroc(pos_score, y.long())
        auprc = self.test_auprc(pos_score, y.long())

        self.log('test_auroc', auroc, on_epoch=True, logger=True)
        self.log('test_auprc', auprc, on_epoch=True, logger=True)

        self.test_step_label.clear()
        self.test_step_preds.clear()  

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(self.parameters(), lr=self.lr, weight_decay=self.wd)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer=optimizer,
            eta_min=0,
            T_max=self.max_epochs
        )
        return {'optimizer': optimizer, 'lr_scheduler': scheduler}

    def get_pretrained_weights(self, model: nn.Module, ckpt_path: str) -> None:
        sd = torch.load(ckpt_path, map_location='cpu', weights_only=False)['state_dict']

        PREFIX = "backbone.roformer."
        DROP_PREFIX = "backbone.cls."   # MLM head; not in RoFormerModel

        remapped = {}
        for k, v in sd.items():
            if k.startswith(DROP_PREFIX):
                continue
            if k.startswith(PREFIX):
                new_k = k[len(PREFIX):]
            else:
                new_k = k
            remapped[new_k] = v

        missing, unexpected = model.load_state_dict(remapped, strict=False)
        print("weights loaded successfully!")
        print("missing keys:", missing)
        print('+'*50)
        print("unexpected keys:", unexpected)

In [ ]:
import wandb
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
def objective(trial: optuna.trial.Trial) -> float:
    try:
        learning_rate = trial.suggest_float("learning_rate", 1e-6, 5e-5)
        weight_decay = trial.suggest_float("weight_decay", 1e-3, 1e-2)
        pooling = trial.suggest_categorical("num_epochs",['cls','mean'])
        use_numeric = trial.suggest_categorical("use_numeric", [True, False])
        
        

        hparams = {'learning_rate': learning_rate,
                   'weight_decay':  weight_decay,
                   'pooling':       pooling,
                   'use_numeric':   use_numeric}
        
        seq_gen = SequencesGenerator(tokenizer_path='../vocab.json',
                                        chunk_length=512,
                                        overlap=0)  
        
        collate_fn = EvalCollator()

        
        train_dataset = EvalDataset(dataset_path='./ehr_arrow_dataset/',
                            data_idx_path='../downstream_idx.parquet',
                            seq_gen=seq_gen,
                            seq_length=512,
                            limits_dict=limits,
                            main_window='within48_query',
                            use_numeric=True,
                            use_time=True,
                            task='y_mort',
                            split='train')
        
        
        val_dataset = EvalDataset(dataset_path='./ehr_arrow_dataset/',
                            data_idx_path='../downstream_idx.parquet',
                            seq_gen=seq_gen,
                            seq_length=512,
                            limits_dict=limits,
                            main_window='within48_query',
                            use_numeric=True,
                            use_time=True,
                            task='y_mort',
                            split='val')
        

        
        train_dataloader = DataLoader(dataset=train_dataset,
                             batch_size=64,
                             num_workers=8,
                             shuffle=True,
                             collate_fn=collate_fn,
                             pin_memory=True,
                             persistent_workers=True,
                             prefetch_factor=4)

        val_dataloader = DataLoader(dataset=val_dataset,
                                     batch_size=64,
                                     num_workers=8,
                                     shuffle=False,
                                     collate_fn=collate_fn,
                                     pin_memory=True,
                                     persistent_workers=True,
                                     prefetch_factor=4)
        
        
        cfg = RoFormerConfig(
            vocab_size=seq_gen.tokenizer.vocab_size,
            hidden_size=768,
            num_hidden_layers=12,
            num_attention_heads=12,
            intermediate_size=3072,
            max_position_embeddings=512,
            pad_token_id=seq_gen.tokenizer.pad_id,
            type_vocab_size= 28,
            visit_vocab_size= 102,
            stage_vocab_size= 5,
            num_labels=2)
        
        
        model = EvalModel(config=cfg,
                            ckpt_path='/scratch/sas10092/ehr-foundation/models/mlm/wandb/run-20251102_081006-Roformer_base_12710830_512_64_25_maskprob_12.5overlap/files/ckpt/epoch=69-step=631120.ckpt',
                            lr=learning_rate,
                            wd=weight_decay,
                            max_epochs=75,
                            pooling=pooling,
                            freeze=False,
                            use_numeric=True,
                            use_time=True)
        
        wandb.login(key='59b6438e0496b3089f91abef35d31dae69b6c009')
        rnd = round(np.random.rand(),4)
        wandb_logger = WandbLogger(project='MedEHR_Eval',
#                         entity=config['logger']['entity'],
                        save_dir='/scratch/sas10092/ehr-foundation/models/hparams',
                        version=f"roformer_512_{rnd}_mortality_within48_query_v5_finetune",
                        name=f"roformer_512_{rnd}_mortality_within48_query_v5_finetune",
#                         tags=[config['version']]
                                  ) 
        
        ckpt_dir = os.path.join(wandb_logger.experiment.dir, 'ckpt')
        os.makedirs(ckpt_dir, exist_ok=True)
        checkpoint_callback = ModelCheckpoint(dirpath=ckpt_dir,
                                              monitor='val_loss', 
                                              mode='min',
                                              every_n_epochs=1,
                                              save_top_k=5)
        
        early_stop = EarlyStopping(monitor='val_loss', 
                        min_delta=0.01,
                        mode='min', 
                        patience=5)
        
        lr_monitor = LearningRateMonitor(logging_interval='epoch')
        
        torch.set_float32_matmul_precision('high')
        trainer = lt.Trainer(accelerator='auto', 
                            devices='auto',
                            strategy='auto',
                            logger=wandb_logger, 
                            log_every_n_steps=1,
                            num_sanity_val_steps=0,
                            max_epochs=5,
                            precision='16-mixed', 
                            callbacks=[early_stop,lr_monitor,checkpoint_callback]
                            )
        trainer.logger.log_hyperparams(hparams)
        trainer.fit(model=model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)


    except optuna.exceptions.TrialPruned:
        wandb.finish()
        raise
    finally:
        wandb.finish()

    return early_stop.best_score.item()





pruner = optuna.pruners.NopPruner()

study = optuna.create_study(study_name='test',
                            direction="minimize", 
                            storage=f'sqlite:////scratch/sas10092/ehr-foundation/models/optuna_dbs/test.db',
                            pruner=pruner,
                            load_if_exists=True,
                            sampler=TPESampler())

study.optimize(objective, n_trials=10,show_progress_bar=True,gc_after_trial=True)

# print("Number of finished trials: {}".format(len(study.trials)))

# print("Best trial:")
# trial = study.best_trial

# print("  Value: {}".format(trial.value))

# print("  Params: ")
# for key, value in trial.params.items():
#     print("    {}: {}".format(key, value))

[I 2025-12-04 14:51:07,077] Using an existing study with name 'test' instead of creating a new one.


  0%|          | 0/10 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/70 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/70 [00:00<?, ?it/s]

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/sas10092/.netrc


...

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [ ]:
4268
614

In [ ]:
# separate the test set full
# 80/20:
# no starstification
# task dependent
# write down about tasks 